# Notebook 3: Evaluate Results

Load trained models, evaluate on all fault types, and plot ROC curves.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve

from config import CONFIG, FAULT_FREQUENCIES
from data_loader import load_paderborn_data
from evaluate import build_test_arrays, evaluate_all_models, get_anomaly_scores, print_results_table
from models import PhysicsAE, DeepSVDD
from utils import load_model

ZIP_PATH = '../data/paderborn-db.zip'

In [ ]:
X_train, X_val, X_fault, scaler = load_paderborn_data(ZIP_PATH, CONFIG)
X_test, y_test = build_test_arrays(X_val, X_fault)

In [ ]:
# Load saved models
device = CONFIG['device']
physics_model  = load_model(PhysicsAE(input_dim=513, latent_dim=32), '../results/models/physicsae.pt', device)
baseline_model = load_model(PhysicsAE(input_dim=513, latent_dim=32), '../results/models/baseline_ae.pt', device)

In [ ]:
# Results table
models = {'PhysicsAE': physics_model, 'Baseline AE': baseline_model}
df = evaluate_all_models(models, X_test, y_test, CONFIG)
print_results_table(df)

In [ ]:
# ROC curves
from sklearn.metrics import roc_auc_score

fig, ax = plt.subplots(figsize=(6, 5))
for name, model in models.items():
    scores = get_anomaly_scores(model, X_test, CONFIG)
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/roc_curves.png', dpi=150)
plt.show()